# Customer and Feeder Clustering

## Objectives

Chapter 2's own "real customers, real diversity" chart already showed
that no two of the 342 real households in this book's own vendored data
follow the same daily rhythm: some barely draw half a kilowatt all day,
others spike past 5kW in the evening. Every study since has treated that
diversity as noise to sample around. This notebook treats it as the
subject: which real, recurring shapes sit underneath that mess, whether
they are stable traits or a one-day snapshot, and whether knowing a
customer's shape actually predicts anything a utility would care about.

By the end you will be able to:

- Cluster real load shapes on their normalized *shape*, not their raw
  magnitude, and check the resulting archetypes with a real validity-
  index comparison rather than an assumed k.
- Recognize a real methodological result: a more advanced clustering
  method (IDEC, a deep autoencoder trained jointly with a clustering
  loss) does not automatically beat a much simpler one, tested here, not
  assumed.
- Measure whether a customer's archetype is a stable trait across a real
  year of data or a snapshot artifact, a question none of this chapter's
  own reference papers test.
- Check whether archetype membership actually predicts which real DER
  constraint (voltage or thermal) a customer's own future PV or EV
  adoption is likely to trigger, using the exact same 31-customer subset
  Chapter 2's own hosting-capacity study already simulated.


## Getting the data

This notebook reuses the same real AusNet smart-meter pool Chapters 1-3
already vendored, no new fetch step needed:

```bash
uv run python scripts/fetch_part4_ausnet_data.py
```


In [ ]:
from pathlib import Path

from lets_plot import LetsPlot
import numpy as np
import pandas as pd

LetsPlot.setup_html()
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")

## Why cluster, not classify

There is no ground-truth "true customer type" label for real households
at this scale, no utility keeps one, so this is a clustering problem, an
unsupervised search for structure, not a classification problem with a
known answer to check against. That changes what "correct" means here:
Chapter 3's phase-identification work had a real answer to check
clustering against; this chapter does not, and has to lean on validity
indices and honest sensitivity checks instead.

The second choice that matters is what "similar" means for a load shape.
Two households can have very different absolute demand, a small
apartment and a large house, and still share the same underlying rhythm,
both peaking in the evening, both quiet at 3am. Clustering on raw
magnitude would separate them by size rather than by behavior. Every
profile below is normalized by its own peak before clustering, the same
"shape, not magnitude" principle Chapter 3 used for a different reason
(voltage correlation, not raw voltage, tracked phase membership); here it
is what lets a small household and a large one land in the same
archetype when their rhythm actually matches.


## Data: a real season, not one day

A single day is a noisy read on anyone's routine, a sick day, a weekend,
a one-off late night. Averaging a real customer's own readings across a
season smooths that out while staying specific to that customer, not a
population average.

In [ ]:
load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

n_customers = load_data.shape[0]


def normalize_shape(profiles: np.ndarray) -> np.ndarray:
    peak = profiles.max(axis=-1, keepdims=True)
    peak = np.where(peak == 0, 1, peak)
    return profiles / peak


season = load_data[:, 0:90, :]
X = normalize_shape(season.mean(axis=1))
print(f"clustering input: {X.shape} (customers, half-hour steps)")

load_data: (342, 365, 48) (customers, days, half-hours)
clustering input: (342, 48) (customers, half-hour steps)


## Baseline: k-means, with k actually checked

The obvious first move is k-means. The question worth checking, not
assuming, is how many clusters. Three validity indices together, an
elbow on inertia, the silhouette coefficient, and the Davies-Bouldin
index, the same combination McLoughlin et al. (2015) and Rajabi et al.
(2020) recommend checking together, since no single index reliably picks
the right k alone.

In [ ]:
from ark.plot.clustering import validity_curve, validity_scores

scores = validity_scores(X, range(2, 10))
scores.round(3)

,k,inertia,silhouette,davies_bouldin
0,2,312.172,0.227,1.572
1,3,265.800,0.190,1.609
2,4,240.341,0.170,1.609
3,5,220.800,0.163,1.649
4,6,207.773,0.160,1.698
5,7,196.869,0.136,1.725
6,8,188.846,0.143,1.652
7,9,181.241,0.136,1.690


In [ ]:
validity_curve(scores)

No sharp elbow, and the silhouette coefficient peaks at k=2 then
decays smoothly, real residential behavior does not split into a small
number of cleanly separated groups, a genuine finding, not a modeling
failure. k=5 is used from here on, past the strict silhouette optimum,
trading a small amount of statistical cleanliness for archetypes still
specific enough to tell a real story with, a defensible, common choice in
applied clustering work, made transparently rather than silently.

In [ ]:
from sklearn.cluster import KMeans

N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
labels = kmeans.fit_predict(X)
pd.Series(labels).value_counts().sort_index()

0     64
1    104
2     74
3     50
4     50
Name: count, dtype: int64

In [ ]:
from ark.plot.clustering import cluster_profiles

cluster_profiles(
    X,
    labels,
    x=np.arange(48) * 0.5,
    x_label="Hour of day",
    y_label="Normalized demand",
    title="Real customer archetypes: member profiles and cluster mean",
)

Five visually distinct rhythms: a sharp early-evening peaker, a
gradual afternoon-into-evening riser, a late-evening peaker with a small
morning bump, and two broadly similar late-night risers, exactly the pair
the modest silhouette score above was warning about. The chart makes
concrete what the number only hinted at: some of these archetypes
genuinely overlap.

## Does a more advanced method earn its complexity?

IDEC (Improved Deep Embedded Clustering, Guo et al. 2017) pretrains an
autoencoder, then jointly optimizes a reconstruction loss and a
clustering loss, in principle learning a representation shaped for
clustering rather than clustering whatever k-means is handed. Kumar and
Mallipeddi (2024) apply the same idea to load-pattern segmentation
directly. Worth testing on this exact data before assuming it helps.

In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score

from ark.cluster.idec import fit_idec

comparison_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=0)
    km_labels = km.fit_predict(X)
    _, idec_labels = fit_idec(X, n_clusters=k, random_state=0)
    comparison_rows.append(
        {
            "k": k,
            "kmeans_silhouette": silhouette_score(X, km_labels),
            "kmeans_davies_bouldin": davies_bouldin_score(X, km_labels),
            "idec_silhouette": silhouette_score(X, idec_labels),
            "idec_davies_bouldin": davies_bouldin_score(X, idec_labels),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.round(3)

,k,kmeans_silhouette,kmeans_davies_bouldin,idec_silhouette,idec_davies_bouldin
0,2,0.227,1.572,0.217,1.609
1,3,0.190,1.609,0.132,2.079
2,4,0.170,1.609,0.078,2.687
3,5,0.163,1.649,0.059,3.132
4,6,0.160,1.698,0.025,3.817
5,7,0.136,1.725,0.025,3.829
6,8,0.143,1.652,0.013,4.554


::: {.ark-mistake}

**IDEC underperforms plain k-means at every k tested, and the gap widens
as k grows.** This is not a bug; it is what happens when a deep method
meets data too small and too low-dimensional to need it: 342 customers,
48 features, is not enough for an autoencoder's extra representational
flexibility to earn its own complexity over a much simpler method already
finding the same structure. The lesson is not "IDEC is bad," it is that a
more advanced method has to prove itself on the data actually in hand,
not get credited for sophistication it does not need here.

:::

## Are these archetypes stable, or a snapshot artifact?

None of this chapter's ten reference papers test this directly: every one
clusters on a single representative period and treats the result as
fixed. AusNet's own real data is a full year, 365 days per customer, and
most of it goes unused once one season is picked. Re-clustering on
different quarters and comparing the result answers a question none of
the ten papers do: is a customer's archetype a stable trait, or does it
drift?

In [ ]:
from sklearn.metrics import adjusted_rand_score

quarters = {"Q1": (0, 90), "Q2": (90, 180), "Q3": (180, 270), "Q4": (270, 360)}
quarter_labels = {}
quarter_silhouettes = {}

for name, (start, end) in quarters.items():
    quarter_x = normalize_shape(load_data[:, start:end, :].mean(axis=1))
    quarter_km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
    quarter_labels[name] = quarter_km.fit_predict(quarter_x)
    quarter_silhouettes[name] = silhouette_score(quarter_x, quarter_labels[name])

pd.Series(quarter_silhouettes, name="silhouette").round(3)

Q1    0.163
Q2    0.159
Q3    0.150
Q4    0.137
Name: silhouette, dtype: float64

Every quarter clusters about as cleanly as the season used above,
so whatever comes next is not an artifact of picking an unusually messy
period. The real test is whether the same customer lands in a matching
archetype across quarters, not just whether each quarter clusters
cleanly on its own.

In [ ]:
quarter_names = list(quarters.keys())
cross_quarter_rows = []
for i in range(len(quarter_names)):
    for j in range(i + 1, len(quarter_names)):
        ari = adjusted_rand_score(quarter_labels[quarter_names[i]], quarter_labels[quarter_names[j]])
        cross_quarter_rows.append({"pair": f"{quarter_names[i]} vs {quarter_names[j]}", "ari": ari})

cross_quarter_df = pd.DataFrame(cross_quarter_rows)
cross_quarter_df.round(3)

,pair,ari
0,Q1 vs Q2,0.128
1,Q1 vs Q3,0.062
2,Q1 vs Q4,0.144
3,Q2 vs Q3,0.336
4,Q2 vs Q4,0.158
5,Q3 vs Q4,0.154


Low, ranging from 0.06 to 0.34. Before trusting that as real drift
rather than the clustering itself just being unstable, it is worth
checking a baseline: how much does re-running k-means on the *same*
quarter, with different random seeds, move the labels around on its own?

In [ ]:
same_quarter_x = normalize_shape(load_data[:, 0:90, :].mean(axis=1))
seed_labels = [
    KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=seed).fit_predict(same_quarter_x) for seed in range(5)
]

seed_ari_rows = []
for i in range(len(seed_labels)):
    for j in range(i + 1, len(seed_labels)):
        ari = adjusted_rand_score(seed_labels[i], seed_labels[j])
        seed_ari_rows.append({"pair": f"seed{i} vs seed{j}", "ari": ari})

pd.DataFrame(seed_ari_rows).round(3)

,pair,ari
0,seed0 vs seed1,0.976
1,seed0 vs seed2,0.967
2,seed0 vs seed3,0.967
3,seed0 vs seed4,0.978
4,seed1 vs seed2,0.979
5,seed1 vs seed3,0.991
6,seed1 vs seed4,0.968
7,seed2 vs seed3,0.970
8,seed2 vs seed4,0.989
9,seed3 vs seed4,0.959


::: {.ark-concept}
<span class="ark-concept-title"><i class="bi bi-info-circle-fill"></i> Key Concept</span>

Same-quarter, different-seed agreement sits at 0.96-0.99, k-means itself
is highly deterministic on this data, ruling out algorithmic noise as the
explanation. Cross-quarter agreement sits at 0.06-0.34, an order of
magnitude lower. The gap between those two numbers is the actual finding:
this is not the clustering method being unreliable, it is real customers
genuinely changing which archetype they belong to as the year moves on,
not a stable trait a utility could safely assume holds from one
assessment to the next.

:::

## Does archetype actually predict DER risk?

Chapter 1 named three ways {{< acr DER >}} strains a feeder, fluctuation,
load growth, and constraint violations, and Chapter 2 found that
{{< acr PV >}} and {{< acr EV >}} adoption bind on *different* specific
constraints on this same feeder, voltage first for {{< acr PV >}},
thermal first for {{< acr EV >}}. If archetype membership predicts which
constraint a customer is likely to trigger, that turns network planning
from reactive, running a hosting-capacity study after a connection
request arrives, into proactive, knowing the risk before it does.

Checkable directly: Chapter 2's own `run_penetration()` sampled its
31-customer subset from this exact 342-customer pool with
`np.random.default_rng(42)`. The same seed reproduces exactly which real
customers that study used, so their archetype labels, from the
342-customer clustering above, and their real simulated risk can be
compared directly, not approximated.

In [ ]:
from ark.dss.circuit import Circuit

TRANSFORMER_KVA = 500.0


def build_ausnet_network() -> Circuit:
    circuit = Circuit.load(DATA_DIR / "LVcircuit-master.txt", solve=False)
    circuit.command("Set VoltageBases = [22.0, 0.400]")
    circuit.command("calcvoltagebases")
    return circuit

In [ ]:
pv_data = np.load(DATA_DIR / "Residential PV data 30-min resolution.npy")


def per_customer_pv_voltage_risk(penetration: int = 100, pv_kva: float = 5.0, day_idx: int = 363, seed: int = 42):
    """Each of Chapter 2's own 31 sampled customers' own worst-case bus voltage under a PV sweep."""
    rng = np.random.default_rng(seed)
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    customer_indices = rng.choice(load_data.shape[0], size=len(loads), replace=False)
    n_with_pv = round(len(loads) * penetration / 100)
    pv_customers = set(rng.choice([load.name for load in loads], size=n_with_pv, replace=False))
    pv_profile = pv_data[day_idx, :]

    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_data[customer_idx, day_idx, :]
        pf = rng.uniform(0.95, 0.99, len(p))
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        if load.name in pv_customers:
            circuit.command(
                f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
                "pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
            )
            circuit.command(
                f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
            )
            circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

    vmax_by_load = {load.name: 0.0 for load in loads}
    for _step in circuit.solve_daily(steps=48):
        voltages = circuit.bus_voltages().set_index("bus")["vmag_pu"]
        for load in loads:
            bus = load.bus1.split(".")[0]
            if bus in voltages.index:
                vmax_by_load[load.name] = max(vmax_by_load[load.name], voltages[bus])

    return customer_indices, [load.name for load in loads], vmax_by_load

In [ ]:
EV_DAY_IDX = int(np.argmax(load_data[:, :, 36:42].sum(axis=(0, 2))))


def per_customer_ev_thermal_contribution(
    adoption_pct: int = 100, ev_shape=None, day_idx: int = EV_DAY_IDX, seed: int = 42
):
    """Each of the same 31 customers' own peak kW draw at the feeder's own peak-loading instant."""
    rng = np.random.default_rng(seed)
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    customer_indices = rng.choice(load_data.shape[0], size=len(loads), replace=False)
    n_with_ev = round(len(loads) * adoption_pct / 100)
    ev_customers = set(rng.choice([load.name for load in loads], size=n_with_ev, replace=False))

    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_data[customer_idx, day_idx, :]
        pf = rng.uniform(0.95, 0.99, len(p))
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        if load.name in ev_customers:
            circuit.command(
                f"New Load.ev_{load.name} bus1={load.bus1} phases=1 kV=(0.4 3 sqrt /) kW=1 pf=0.98 "
                "model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=variable"
            )
            circuit.command(
                f"New LoadShape.ev_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={ev_shape.tolist()}"
            )
            circuit.command(f"edit Load.ev_{load.name} daily=ev_profile_{load.name}")

    utilization_by_step = []
    kw_by_step = []
    for _step in circuit.solve_daily(steps=48):
        transformer_power = circuit.element_powers("transformers")
        s_kva = np.sqrt(transformer_power["p_kw"] ** 2 + transformer_power["q_kvar"] ** 2).sum()
        utilization_by_step.append(s_kva / TRANSFORMER_KVA * 100)
        kw_by_step.append(circuit.element_powers("loads").set_index("name")["p_kw"].to_dict())

    peak_step = int(np.argmax(utilization_by_step))
    peak_kw = kw_by_step[peak_step]
    combined = {
        load.name: peak_kw.get(load.name.lower(), 0.0) + peak_kw.get(f"ev_{load.name}".lower(), 0.0) for load in loads
    }
    return customer_indices, [load.name for load in loads], combined

Both scenarios use exactly the same seed and the same sampling call
as Chapter 2's own study, so they draw the same 31 real customers.

In [ ]:
customer_idx_pv, load_names, vmax_by_load = per_customer_pv_voltage_risk()

uncoordinated_shape = np.zeros(48)
uncoordinated_shape[36:42] = 7.36  # 18:00-21:00, full Level-2 rating
customer_idx_ev, load_names_ev, kw_by_load = per_customer_ev_thermal_contribution(ev_shape=uncoordinated_shape)

if list(customer_idx_pv) != list(customer_idx_ev) or load_names != load_names_ev:
    raise ValueError("PV and EV scenarios sampled different customers, seeds should match")

archetype_for_31 = labels[customer_idx_pv]
risk_df = pd.DataFrame(
    {
        "load_name": load_names,
        "archetype": archetype_for_31,
        "vmax_pu": [vmax_by_load[name] for name in load_names],
        "peak_kw_contribution": [kw_by_load[name] for name in load_names],
    }
)
risk_by_archetype = risk_df.groupby("archetype")[["vmax_pu", "peak_kw_contribution"]].agg(["mean", "count"])
risk_by_archetype.round(4)

vmax_pu       peak_kw_contribution      
             mean count                 mean count
archetype                                         
0          1.0980     7               8.4303     7
1          1.0970     5               9.8764     5
2          1.0994    10              10.6108    10
3          1.1019     4               8.3780     4
4          1.0956     5               8.0516     5

::: {.ark-concept}
<span class="ark-concept-title"><i class="bi bi-info-circle-fill"></i> Key Concept</span>

Archetype membership carries real signal, and it does not point the same
way for both {{< acr DER >}} types. The archetype with the highest mean
voltage risk is not the same archetype with the highest mean thermal
contribution, mirroring Chapter 2's own PV-vs-EV finding at the archetype
level: {{< acr PV >}} risk and {{< acr EV >}} risk concentrate in
different customer groups, not the same one. A DSO that only checks
"which archetype is risky" without asking "risky for which constraint"
would still miss half the picture. The sample here is small, 31
customers across five archetypes, so this is a real, checkable
association worth pairing with more data before treating it as settled,
not a large-scale statistical claim.

:::